In [ ]:
!pip install ultralytics -q
print("Ultralytics is installed")

In [ ]:
import torch
assert torch.cuda.is_available()
print(f"GPU: {torch.cuda.get_device_name(0)}")
print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory/1e9:.1f} GB")

In [ ]:
# Params for the first run
from pathlib import Path

DATASET_DIR  = "/kaggle/input/datasets/natair/chicken-dataset-v2/chicken_dataset_v2"
WORK_DIR     = "/kaggle/working"
RUN_NAME     = "chicken_detector_v2"
IMG_SIZE     = 1280
TOTAL_EPOCHS = 150
BATCH        = 8
MODEL_BASE   = "yolov8m.pt"

In [ ]:
# Params for fine-tuning
from ultralytics import YOLO
import os
from pathlib import Path

DATASET_DIR         = "/kaggle/input/datasets/natair/chicken-dataset-v3-845/chicken_dataset_v3"
WORK_DIR            = "/kaggle/working"
RUN_NAME            = "chicken_detector_v3"
IMG_SIZE            = 1280
TOTAL_EPOCHS        = 60
BATCH               = 8
FINETUNE_BASE_MODEL = "/kaggle/input/models/natair/chicken-label-drone-model-tolo8m-825frames/pytorch/default/1/best_l.pt"

In [ ]:
# Creating a dataset config for YOLO (first run)
data_yaml = f"""
path: {DATASET_DIR}
train: images/train
val: images/val

names:
  0: chicken
"""

yaml_path = f"{WORK_DIR}/data.yaml"
with open(yaml_path, "w") as f:
    f.write(data_yaml)

print(f"data.yaml created: {yaml_path}")
print(data_yaml)

train_imgs = list(Path(DATASET_DIR, "images/train").glob("*.jpg"))
val_imgs   = list(Path(DATASET_DIR, "images/val").glob("*.jpg"))
train_lbls = list(Path(DATASET_DIR, "labels/train").glob("*.txt"))
val_lbls   = list(Path(DATASET_DIR, "labels/val").glob("*.txt"))

print(f"Train: {len(train_imgs)} images / {len(train_lbls)} annotaions")
print(f"Val:   {len(val_imgs)} images / {len(val_lbls)} annotaions")

assert len(train_imgs) > 0, "No train images found."
assert len(train_imgs) == len(train_lbls), "The number of images and annotations does not match!"


In [ ]:
# Creating a dataset config for YOLO (fine-tuning)
yaml_path      = "/kaggle/working/chicken_dataset_v3_boosted/data.yaml"
DATASET_DIR_v2 = "/kaggle/working/chicken_dataset_v3_boosted"

train_imgs = list(Path(DATASET_DIR_v2, "images/train").glob("*.jpg"))
val_imgs   = list(Path(DATASET_DIR_v2, "images/val").glob("*.jpg"))
train_lbls = list(Path(DATASET_DIR_v2, "labels/train").glob("*.txt"))
val_lbls   = list(Path(DATASET_DIR_v2, "labels/val").glob("*.txt"))

print(f"Train: {len(train_imgs)} images / {len(train_lbls)} annotaions")
print(f"Val:   {len(val_imgs)} images / {len(val_lbls)} annotaions")

assert len(train_imgs) > 0, "No train images found."
assert len(train_imgs) == len(train_lbls), "The number of images and annotations does not match!"


In [ ]:
# First training run
from ultralytics import YOLO
import os

RESUME_CHECKPOINT = None
# "/kaggle/input/chicken-checkpoint/last.pt"

if RESUME_CHECKPOINT and Path(RESUME_CHECKPOINT).exists():
    print(f"Continuing training from the checkpoint: {RESUME_CHECKPOINT}")
    model = YOLO(RESUME_CHECKPOINT)
    resume_flag = True
else:
    print(f"Start training from scratch: {MODEL_BASE}")
    model = YOLO(MODEL_BASE)
    resume_flag = False

results = model.train(
    data=yaml_path,
    epochs=TOTAL_EPOCHS,
    imgsz=IMG_SIZE,
    batch=BATCH,
    name=RUN_NAME,
    project=f"{WORK_DIR}/runs",
    resume=resume_flag,

    # Augmentations for small objects
    mosaic=1.0,
    scale=0.9,
    copy_paste=0.3,
    degrees=10.0,
    fliplr=0.5,
    flipud=0.3,
    hsv_h=0.015,
    hsv_v=0.4,

    # Session interruption resilience
    save=True,
    save_period=5,      # checkpoint every 5 epochs
    patience=40,        # early discontinuation if there is no improvement over a prolonged period
    exist_ok=True,      # write to the same `run` folder again
    verbose=True,
    plots=True,
)

print("\n Training completed (or interrupted due to patience)")
print(f"  best.pt: {WORK_DIR}/runs/{RUN_NAME}/weights/best.pt")
print(f"  last.pt: {WORK_DIR}/runs/{RUN_NAME}/weights/last.pt")

In [ ]:
# Fine-tuning run
RESUME_CHECKPOINT = None
# "/kaggle/input/chicken-checkpoint-v3/last.pt"

if RESUME_CHECKPOINT and Path(RESUME_CHECKPOINT).exists():
    print(f"Continuing interrupted fine-tuning from: {RESUME_CHECKPOINT}")
    model = YOLO(RESUME_CHECKPOINT)
    resume_flag = True
else:
    print(f"Starting fine-tuning from: {FINETUNE_BASE_MODEL}")
    model = YOLO(FINETUNE_BASE_MODEL)
    resume_flag = False

results = model.train(
    data=yaml_path,
    epochs=80,
    imgsz=IMG_SIZE,
    batch=BATCH,
    name="chicken_detector_v3_boosted",
    project=f"{WORK_DIR}/runs",
    resume=resume_flag,
    lr0=0.005,
    lrf=0.01,
    mosaic=1.0,
    scale=0.9,
    copy_paste=0.3,
    degrees=10.0,
    fliplr=0.5,
    flipud=0.3,
    hsv_h=0.015,
    hsv_v=0.4,
    save=True,
    save_period=5,
    patience=50,
    exist_ok=True,
    verbose=True,
    plots=True,
)

print("\n Fine-tuning completed (or interrupted due to patience)")
print(f"  best.pt: {WORK_DIR}/runs/chicken_detector_v3_boosted/weights/best.pt")
print(f"  last.pt: {WORK_DIR}/runs/chicken_detector_v3_boosted/weights/last.pt")

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

results_csv = f"{WORK_DIR}/runs/{RUN_NAME}_boosted/results.csv"
df = pd.read_csv(results_csv)
df.columns = df.columns.str.strip()

print("Last 5 epochs:")
print(df[["epoch", "metrics/precision(B)", "metrics/recall(B)",
          "metrics/mAP50(B)", "metrics/mAP50-95(B)"]].tail())

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].plot(df["epoch"], df["metrics/mAP50(B)"], label="mAP50")
axes[0].plot(df["epoch"], df["metrics/mAP50-95(B)"], label="mAP50-95")
axes[0].set_title("mAP by epochs")
axes[0].legend(); axes[0].grid(alpha=0.3)

axes[1].plot(df["epoch"], df["metrics/precision(B)"], label="Precision")
axes[1].plot(df["epoch"], df["metrics/recall(B)"], label="Recall")
axes[1].set_title("Precision / Recall")
axes[1].legend(); axes[1].grid(alpha=0.3)

plt.tight_layout()
plt.savefig(f"{WORK_DIR}/training_summary.jpg", dpi=120)
plt.show()

best_map50 = df["metrics/mAP50(B)"].max()
print(f"\nBest mAP50: {best_map50:.3f}")

In [ ]:
best_model = YOLO(f"{WORK_DIR}/runs/{RUN_NAME}_boosted/weights/best.pt")

# Picking any image from 'val' for a visual check.
test_img = str(val_imgs[0])
res = best_model(test_img, imgsz=IMG_SIZE, conf=0.25)

res[0].save(filename=f"{WORK_DIR}/test_prediction.jpg")
print(f"Result saved: {WORK_DIR}/test_prediction.jpg")
print(f"Objects found: {len(res[0].boxes)}")

import cv2
img = cv2.cvtColor(cv2.imread(f"{WORK_DIR}/test_prediction.jpg"), cv2.COLOR_BGR2RGB)
plt.figure(figsize=(16, 9))
plt.imshow(img)
plt.axis("off")
plt.title(f"Detections: {len(res[0].boxes)}")
plt.show()

In [ ]:
from ultralytics import YOLO
import matplotlib.pyplot as plt
import cv2

model_v2 = YOLO("/kaggle/input/models/natair/chicken-label-drone-model-tolo8m-825frames/pytorch/default/1/best_l.pt")
model_v3 = YOLO("/kaggle/working/runs/chicken_detector_v3_boosted/weights/best.pt")

results_per_img = []
for fp in val_imgs:
    res_v3 = model_v3(str(fp), imgsz=1280, conf=0.25, iou=0.35, verbose=False)
    results_per_img.append((fp, len(res_v3[0].boxes)))

results_per_img.sort(key=lambda x: -x[1])
top_cluster_frames = [fp for fp, n in results_per_img[:6]]

print("Shots with the largest number of chickens (potential candidates for the groups):")
for fp, n in results_per_img[:6]:
    print(f"  {fp.name}: {n} chickens (v3)")

# Comparison of v2 vs v3 in these frames
fig, axes = plt.subplots(2, len(top_cluster_frames), figsize=(6*len(top_cluster_frames), 10))

for col, fp in enumerate(top_cluster_frames):
    img_orig = cv2.imread(str(fp))

    res_v2 = model_v2(str(fp), imgsz=1280, conf=0.25, iou=0.35, verbose=False)
    img_v2 = cv2.cvtColor(img_orig.copy(), cv2.COLOR_BGR2RGB)
    for box in res_v2[0].boxes.xyxy.cpu().numpy():
        x1, y1, x2, y2 = map(int, box)
        cv2.rectangle(img_v2, (x1, y1), (x2, y2), (255, 0, 0), 2)
    axes[0, col].imshow(img_v2)
    axes[0, col].set_title(f"v2: {len(res_v2[0].boxes)} chickens", fontsize=10)
    axes[0, col].axis("off")

    res_v3 = model_v3(str(fp), imgsz=1280, conf=0.25, iou=0.35, verbose=False)
    img_v3 = cv2.cvtColor(img_orig.copy(), cv2.COLOR_BGR2RGB)
    for box in res_v3[0].boxes.xyxy.cpu().numpy():
        x1, y1, x2, y2 = map(int, box)
        cv2.rectangle(img_v3, (x1, y1), (x2, y2), (0, 255, 0), 2)
    axes[1, col].imshow(img_v3)
    axes[1, col].set_title(f"v3: {len(res_v3[0].boxes)} chickens", fontsize=10)
    axes[1, col].axis("off")

plt.tight_layout()
plt.savefig("/kaggle/working/v2_vs_v3_clusters.jpg", dpi=100, bbox_inches="tight")
plt.show()

In [ ]:
import os

old_dir = "/kaggle/input/datasets/natair/chicken-dataset-v2/chicken_dataset_v2/labels/train"
new_dir = "/kaggle/input/datasets/natair/chicken-dataset-v3-845/chicken_dataset_v3/labels/train"

changed = 0
checked = 0
for fname in os.listdir(new_dir):
    old_path = os.path.join(old_dir, fname)
    new_path = os.path.join(new_dir, fname)
    if os.path.isfile(old_path) and os.path.isfile(new_path):
        checked += 1
        old_n = sum(1 for _ in open(old_path))
        new_n = sum(1 for _ in open(new_path))
        if new_n > old_n:
            changed += 1

print(f"Files checked: {checked}")
print(f"Images with added boxes: {changed}")

In [ ]:
# Duplicating Frames with Additional Markup of Clusters
import os
import shutil
from pathlib import Path

OLD_LABELS_DIR = "/kaggle/input/datasets/natair/chicken-dataset-v2/chicken_dataset_v2/labels/train"
NEW_LABELS_DIR = "/kaggle/input/datasets/natair/chicken-dataset-v3-845/chicken_dataset_v3/labels/train"
NEW_IMAGES_DIR = "/kaggle/input/datasets/natair/chicken-dataset-v3-845/chicken_dataset_v3/images/train"

OUT_IMAGES_DIR = "/kaggle/working/chicken_dataset_v3_boosted/images/train"
OUT_LABELS_DIR = "/kaggle/working/chicken_dataset_v3_boosted/labels/train"

N_DUPLICATES = 3

os.makedirs(OUT_IMAGES_DIR, exist_ok=True)
os.makedirs(OUT_LABELS_DIR, exist_ok=True)

changed_files = []
checked = 0

for fname in os.listdir(NEW_LABELS_DIR):
    old_path = os.path.join(OLD_LABELS_DIR, fname)
    new_path = os.path.join(NEW_LABELS_DIR, fname)
    if os.path.isfile(old_path) and os.path.isfile(new_path):
        checked += 1
        old_n = sum(1 for _ in open(old_path))
        new_n = sum(1 for _ in open(new_path))
        if new_n > old_n:
            changed_files.append(fname)

print(f"Checked: {checked}  |  Changed (clusters have been relabeled): {len(changed_files)}")

copied_base = 0
for fname in os.listdir(NEW_LABELS_DIR):
    label_src = os.path.join(NEW_LABELS_DIR, fname)
    img_name = Path(fname).stem + ".jpg"
    img_src = os.path.join(NEW_IMAGES_DIR, img_name)

    if not os.path.isfile(img_src):
        print(f"No image found for {fname}")
        continue

    shutil.copy(label_src, os.path.join(OUT_LABELS_DIR, fname))
    shutil.copy(img_src, os.path.join(OUT_IMAGES_DIR, img_name))
    copied_base += 1

print(f"The base train has been copied: {copied_base} pairs")

duplicated = 0
for fname in changed_files:
    stem = Path(fname).stem
    label_src = os.path.join(NEW_LABELS_DIR, fname)
    img_src = os.path.join(NEW_IMAGES_DIR, stem + ".jpg")

    if not os.path.isfile(img_src):
        continue

    for dup_idx in range(1, N_DUPLICATES + 1):
        new_stem = f"{stem}_dup{dup_idx}"
        shutil.copy(label_src, os.path.join(OUT_LABELS_DIR, new_stem + ".txt"))
        shutil.copy(img_src, os.path.join(OUT_IMAGES_DIR, new_stem + ".jpg"))
        duplicated += 1

print(f" Duplicates have been added: {duplicated}")
print(f" Final train: {copied_base + duplicated} files")
print(f" (was {copied_base}, clustered frames are now found in {N_DUPLICATES + 1}x)")

In [ ]:
import shutil

archive_path = shutil.make_archive(
    "/kaggle/working/chicken_dataset_v3_boosted",
    "zip",
    "/kaggle/working/chicken_dataset_v3_boosted"
)

import os
size_mb = os.path.getsize(archive_path) / 1e6
print(f" The archive was created: {archive_path}")
print(f" Size: {size_mb:.1f} MB")

In [ ]:
VAL_LABELS_DIR = "/kaggle/input/datasets/natair/chicken-dataset-v3-845/chicken_dataset_v3/labels/val"
VAL_IMAGES_DIR = "/kaggle/input/datasets/natair/chicken-dataset-v3-845/chicken_dataset_v3/images/val"

OUT_VAL_IMAGES_DIR = "/kaggle/working/chicken_dataset_v3_boosted/images/val"
OUT_VAL_LABELS_DIR = "/kaggle/working/chicken_dataset_v3_boosted/labels/val"

os.makedirs(OUT_VAL_IMAGES_DIR, exist_ok=True)
os.makedirs(OUT_VAL_LABELS_DIR, exist_ok=True)

for fname in os.listdir(VAL_LABELS_DIR):
    stem = Path(fname).stem
    shutil.copy(os.path.join(VAL_LABELS_DIR, fname), os.path.join(OUT_VAL_LABELS_DIR, fname))
    shutil.copy(os.path.join(VAL_IMAGES_DIR, stem + ".jpg"), os.path.join(OUT_VAL_IMAGES_DIR, stem + ".jpg"))

print(f" val was copied as-is: {len(os.listdir(VAL_LABELS_DIR))} files")

data_yaml_content = f"""
path: /kaggle/working/chicken_dataset_v3_boosted
train: images/train
val: images/val

names:
  0: chicken
"""
with open("/kaggle/working/chicken_dataset_v3_boosted/data.yaml", "w") as f:
    f.write(data_yaml_content)
print(" data.yaml created within the dataset")